# Cell Tower Optimization — End-to-End Demo

This notebook demonstrates the full AI-driven cell tower placement pipeline for Nagpur, India.
It runs a **fast configuration** (reduced generations) so you can see results in under 10 minutes
on a standard laptop without a GPU.

## What this notebook shows
1. Data loading and feature engineering (population, OSM, elevation)
2. Random Forest demand prediction — with real OpenCellID labels if available
3. NSGA-II multi-objective optimization producing a Pareto frontier
4. Interactive Folium coverage map with COST-231 Hata propagation circles
5. Comparison table: AI solution vs. real OpenCellID network

---
**Prerequisites**: `pip install -r requirements.txt` (from `upgraded cell tower optimization/`)

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Change to the main pipeline directory so imports resolve correctly
PIPELINE_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'upgraded cell tower optimization')
if os.path.exists(PIPELINE_DIR):
    os.chdir(PIPELINE_DIR)
else:
    # Already in the right place if running from repo root
    os.chdir('upgraded cell tower optimization')

print('Working directory:', os.getcwd())

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from IPython.display import display, IFrame

from src.data.preprocessing import DataPreprocessor
from src.models.demand import DemandModel
from src.optimization.candidates import CandidateSiteGenerator
from src.optimization.nsga2 import MultiObjectiveOptimizer
from src.visualization.visualize import Visualizer

print('All imports successful.')

In [ ]:
# --- Configuration ---
# Override NSGA-II settings for a fast demo run
with open('src/config/config.yaml') as f:
    config = yaml.safe_load(f)

config['nsga2']['pop_size'] = 30   # reduced for demo speed
config['nsga2']['n_gen']    = 10   # reduced for demo speed
config['optimization']['num_towers'] = 15

print('Config loaded. NSGA-II: pop=30, gen=10 (fast demo mode)')
print(f"  Frequency : {config['rf_params']['frequency_mhz']} MHz")
print(f"  Tx Power  : {config['rf_params']['transmit_power_dbm']} dBm")
print(f"  Antenna h : {config['rf_params']['antenna_height_m']} m")

## Step 1 — Data Loading & Feature Engineering

In [ ]:
preprocessor = DataPreprocessor(config)
boundary = preprocessor.create_nagpur_boundary()
grid = preprocessor.generate_planning_grid(boundary)
print(f'Planning grid: {len(grid):,} cells at {config["optimization"]["grid_resolution"]} m resolution')

buildings, roads, landuse = preprocessor.extract_osm_features(boundary)
grid_features = preprocessor.compute_grid_features(
    grid, buildings, roads, landuse,
    config['paths']['pop_tif'], config['paths']['dem_tif']
)
print('Feature columns:', list(grid_features.columns))

## Step 2 — ML Demand Prediction

In [ ]:
opencellid_path = os.path.join('Data', 'raw', 'opencellid_nagpur.csv')

demand_model = DemandModel(
    config,
    opencellid_csv=opencellid_path if os.path.exists(opencellid_path) else None
)
demand_model.train_ml_model(grid_features)

print('\nLabel source:', demand_model.label_source)

In [ ]:
grid_with_demand = demand_model.predict_traffic(grid_features)
grid_with_demand, high_demand, hotspots = demand_model.detect_hotspots(grid_with_demand)

# Feature importance plot
features = ['population', 'road_density', 'building_density', 'urban_class']
importances = demand_model.model.feature_importances_
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(features, importances, color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importance for Demand Prediction')
plt.tight_layout()
plt.show()

## Step 3 — Candidate Site Generation

In [ ]:
generator = CandidateSiteGenerator(config)
candidates = generator.generate_candidates(grid_with_demand)
print(f'Candidates generated: {len(candidates)}')

## Step 4 — NSGA-II Multi-Objective Optimization

In [ ]:
optimizer = MultiObjectiveOptimizer(config)
res, best_config = optimizer.run_optimization(candidates, grid_with_demand)
print(f'\nSelected {len(best_config)} tower locations from Pareto front')

In [ ]:
# Pareto Frontier
F = res.F
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(-F[:, 0], -F[:, 1], c=F[:, 2], cmap='viridis', s=50)
plt.colorbar(sc, ax=ax, label='Deployment Cost (lower = better)')
ax.set_xlabel('Satisfied Demand (Weighted Coverage)')
ax.set_ylabel('Network Capacity — Sum Throughput (Mbps)')
ax.set_title('NSGA-II Pareto Frontier\n(each point = a valid non-dominated tower configuration)')
plt.tight_layout()
plt.show()

## Step 5 — Coverage Map (COST-231 Hata propagation radius)

In [ ]:
visuals = Visualizer(config)
visuals.generate_html_map(best_config, grid_with_demand, boundary)
map_path = os.path.join(config['paths']['output_dir'], 'interactive_towers.html')
IFrame(src=map_path, width='100%', height=500)

## Step 6 — Results Comparison

If real OpenCellID data is present, compare the AI solution against the real network.

In [ ]:
from src.models.propagation import PropagationModel
import numpy as np

def compute_coverage_stats(tower_df, grid_df, config):
    """Compute population coverage % and tower count for a given set of towers."""
    crs = config['project']['crs']
    grid_res = config['optimization']['grid_resolution']
    f_mhz = config['rf_params']['frequency_mhz']
    tx_dbm = config['rf_params']['transmit_power_dbm']
    rsrp_min = -110  # dBm

    # Max coverage radius (metres)
    h_te  = config['rf_params']['antenna_height_m']
    h_re  = config['rf_params'].get('receiver_height_m', 1.5)
    a_hre = (1.1 * np.log10(f_mhz) - 0.7) * h_re - (1.56 * np.log10(f_mhz) - 0.8)
    intercept = 46.3 + 33.9 * np.log10(f_mhz) - 13.82 * np.log10(h_te) - a_hre + 3
    slope = 44.9 - 6.55 * np.log10(h_te)
    max_loss = tx_dbm - rsrp_min
    d_km = float(np.clip(10 ** ((max_loss - intercept) / slope), 0.2, 5.0))
    radius_m = d_km * 1000.0

    tower_xy = tower_df[['x', 'y']].values
    grid_xy  = grid_df[['x', 'y']].values
    pop      = grid_df['population'].values

    covered = np.zeros(len(grid_df), dtype=bool)
    for tx, ty in tower_xy:
        dists = np.sqrt((grid_xy[:, 0] - tx)**2 + (grid_xy[:, 1] - ty)**2)
        covered |= (dists <= radius_m)

    total_pop = pop.sum()
    cov_pop = pop[covered].sum()
    return {
        'num_towers': len(tower_df),
        'coverage_pct': (covered.sum() / len(grid_df)) * 100,
        'population_coverage_pct': (cov_pop / total_pop * 100) if total_pop > 0 else 0,
        'coverage_radius_m': radius_m
    }

ai_stats = compute_coverage_stats(best_config, grid_with_demand, config)

results = {
    'AI Optimized': ai_stats
}

opencellid_path = os.path.join('Data', 'raw', 'opencellid_nagpur.csv')
if os.path.exists(opencellid_path):
    oc = pd.read_csv(opencellid_path)
    if 'radio' in oc.columns:
        oc = oc[oc['radio'].str.upper() == 'LTE']
    import pyproj
    lon_col = 'lon' if 'lon' in oc.columns else 'longitude'
    lat_col = 'lat' if 'lat' in oc.columns else 'latitude'
    oc_gdf = gpd.GeoDataFrame(oc, geometry=gpd.points_from_xy(oc[lon_col], oc[lat_col]), crs='EPSG:4326')
    oc_gdf = oc_gdf.to_crs(config['project']['crs'])
    real_df = pd.DataFrame({'x': oc_gdf.geometry.x, 'y': oc_gdf.geometry.y})
    real_stats = compute_coverage_stats(real_df, grid_with_demand, config)
    results['Real Network (OpenCellID)'] = real_stats
else:
    print('OpenCellID data not available — showing AI results only.')
    print('Download from opencellid.org (MCC=404, filter LTE, Nagpur bbox) to enable comparison.')

# Print table
print(f"\n{'Metric':<35} ", end='')
for k in results: print(f"{k:<30} ", end='')
print()
print('-' * (35 + 30 * len(results)))
for metric in ['num_towers', 'coverage_pct', 'population_coverage_pct', 'coverage_radius_m']:
    label = {
        'num_towers': 'Number of towers',
        'coverage_pct': 'Area coverage (%)',
        'population_coverage_pct': 'Population coverage (%)',
        'coverage_radius_m': 'Coverage radius (m, COST-231)'
    }[metric]
    print(f"{label:<35} ", end='')
    for v in results.values():
        val = v[metric]
        print(f"{val:<30.1f} ", end='')
    print()

---
## Summary

The table above is the core claim of this project: the AI system recommends a smaller number of
towers while maintaining or improving population coverage, compared to the real deployed network.

Key technical choices:
- **COST-231 Hata**: Industry-standard urban propagation model (not a fixed circle)
- **NSGA-II**: True multi-objective optimisation — no arbitrary single-objective weighting
- **Random Forest demand model**: Trained on real OpenCellID density (when available) or
  population+OSM proxy (disclosed as proxy, not measured throughput)
- **Validation**: Hungarian bipartite matching against real tower coordinates, not mocked data